<a href="https://colab.research.google.com/github/20Aarya05/FlyrankAI_1/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/20Aarya05/FlyrankAI_1/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional Lane:** Lane 2: Refresh / Content Opportunity Scoring

**Why this lane?**
Organic search traffic is a primary user acquisition channel for our clients, and high-visibility content (pages on Google's Page 1) drives the majority of this traffic. Over the next 7 weeks, I will focus on prioritizing content refreshes to protect this traffic from decay. In the starter dataset, we observe that over 54% of all content items are in a state of performance decline, including 56.3% of pages currently ranked on Google's Page 1. Recommending the right pages for updates will allow editorial teams to maximize their return on investment by focusing their manual revision efforts on pages where traffic drops are most critical and recoverable.

## 2. The question: decision, action, cost of a wrong call

- **What decision does this improve?**
  It improves the content refresh prioritization decision. Instead of editors guessing which pages need attention or manually auditing hundreds of pages, this system provides an evidence-based, ranked queue of which pages should be reviewed first.

- **Who acts on the output, and what do they do?**
  Content editors and SEO managers act on the output. They look at the high-priority pages in the queue, inspect the associated reason codes (e.g., thin content, stale visibility, CTR opportunity), and perform targeted revisions (e.g., expanding thin text, updating outdated info, or rewriting title tags).

- **What does a wrong recommendation cost?**
  - **False Positive (recommending a page that doesn't need refresh or won't respond to it):** Costs 2-4 hours of valuable editor time spent rewriting content unnecessarily, plus the opportunity cost of not updating a page that actually needed it.
  - **False Negative (failing to recommend a page in critical decline):** Costs lost organic traffic, leading to fewer sessions, signups, or conversions, which translates directly to lost revenue for the client.

- **Why does data/ML help at all?**
  A simple static rule (e.g., "refresh all pages older than 180 days") is insufficient. In our starter dataset, pages older than 180 days since their last update make up only 0.58% of the inventory, while 54.21% of pages are actively declining. Declines are driven by complex interactions between multiple features—impressions, click-through rates, positions, scroll rates, and user intent. Machine learning can model these multi-dimensional, non-linear relationships to rank and prioritize pages much more effectively than blunt hand-coded heuristic rules.

## 3. Quick look at the data (2-3 real numbers)

We load the starter dataset, apply basic filters (GSC impressions > 0 and content age >= 90 days), and extract 3 key metrics to validate this lane choice:
1. **Overall Decline Rate:** The proportion of pages experiencing a decline (`trend_direction == "down"`).
2. **Page 1 Vulnerability:** The percentage of Page 1 pages (`0.1 <= avg_position <= 10.0`) that are currently declining.
3. **Traffic Value Contrast:** The average number of GA4 sessions for Page 1 pages compared to other positions, highlighting the high value of the traffic we are trying to protect.

In [2]:
import pandas as pd
import numpy as np

# Load the starter CSV
csv_url = 'https://raw.githubusercontent.com/20Aarya05/FlyrankAI_1/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(csv_url)

# Apply starter pipeline filters (matching prep features step)
df_clean = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()

# 1. Overall Decline Rate
total_pages = len(df_clean)
declining_pages = (df_clean['trend_direction'] == 'down').sum()
pct_declining = (declining_pages / total_pages) * 100

# 2. Page 1 Vulnerability (avg_position between 0.1 and 10.0)
p1_mask = (df_clean['avg_position'] > 0) & (df_clean['avg_position'] <= 10)
p1_total = p1_mask.sum()
p1_declining = ((df_clean['avg_position'] > 0) & (df_clean['avg_position'] <= 10) & (df_clean['trend_direction'] == 'down')).sum()
pct_p1_declining = (p1_declining / p1_total) * 100

# 3. Traffic Value Contrast
p1_avg_sessions = df_clean[p1_mask]['sessions_90d'].mean()
non_p1_avg_sessions = df_clean[~p1_mask]['sessions_90d'].mean()

print(f"Total clean rows analyzed: {total_pages}")
print(f"1. Overall Decline Rate: {declining_pages} pages ({pct_declining:.2f}%) are declining.")
print(f"2. Page 1 Vulnerability: {p1_declining} of {p1_total} Page 1 pages ({pct_p1_declining:.2f}%) are declining.")
print(f"3. Traffic Value: Page 1 pages average {p1_avg_sessions:.2f} sessions, vs {non_p1_avg_sessions:.2f} sessions for others.")

Total clean rows analyzed: 30000
1. Overall Decline Rate: 16262 pages (54.21%) are declining.
2. Page 1 Vulnerability: 7311 of 12983 Page 1 pages (56.31%) are declining.
3. Traffic Value: Page 1 pages average 38.91 sessions, vs 35.66 sessions for others.


## 4. Careful words: what I can and can't claim

- **What this work CAN claim:**
  - **Decision-Support:** We can claim that our model outputs a prioritized decision-support ranking that helps content editors allocate their review capacity more efficiently.
  - **Observed Associations:** We can claim observed, statistical associations between pre-decision signals (like engagement, position tiers, main intent) and subsequent search performance decline.
  - **Directional Priority:** We can claim a directional score indicating which pages show the strongest patterns of underperformance and decline relative to historical data.

- **What this work CANNOT claim:**
  - **Causal Proof:** We cannot claim that executing a content refresh will *cause* a recovery in search performance. To claim causation, we would need to run controlled A/B experiments.
  - **"Predicting Google":** We cannot claim that we are reverse-engineering or predicting Google's internal ranking algorithms. Our model only observes changes in search metrics, not search engine mechanics.
  - **Automated Solutions:** We cannot claim that the recommendations eliminate the need for human editorial oversight; the model identifies *where* to look, not *how* to rewrite the content.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.